# MATCARE-AI: CBC Anemia Analysis, Population Audit & Hybrid Modeling
This notebook executes Phase 1 population and demographic auditing (including verifying the 473 female non-anemic sample cohort),
data leakage analysis, stratified splitting, base models, adaptive hybrid optimization, and SHAP explainability.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')
from preprocessing.anemia_preprocessor import load_and_inspect_anemia, audit_cbc_population_and_leakage, get_anemia_split
from training.hybrid_model import AdaptiveWeightedHybridClassifier
from evaluation.metrics import compare_four_approaches
from explainability.shap_explainers import compute_tree_shap_importances, compute_hybrid_global_importances

## 1. Population Audit (Verifying the 473 Sample Finding)

In [ ]:
data_path = os.path.join('..', 'datasets', 'anemia_cbc.csv')
df_an = load_and_inspect_anemia(data_path)
audit = audit_cbc_population_and_leakage(df_an)
print('Total Rows:', audit['total_rows'])
print('Gender Counts:', audit['gender_distribution'])
print('Female Non-Anemic Cohort (473 Check):', audit['female_non_anemic_count'])
print('\nPopulation Research Note:\n', audit['population_finding'])
print('\nHemoglobin-Result Correlation:', audit['hb_result_correlation'])

## 2. Train/Val/Test Split & Hybrid Model Execution

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, scaler, metadata = get_anemia_split(df_an)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

xgb = XGBClassifier(n_estimators=120, max_depth=5, learning_rate=0.08, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)

hybrid = AdaptiveWeightedHybridClassifier(rf_model=rf, xgb_model=xgb, classes=np.array([0, 1]))
opt_res = hybrid.optimize_weights(X_val, y_val, metric='macro_f1', high_risk_idx=1)
print(f'Optimal Weights: w_RF={opt_res["best_w_rf"]:.4f}, w_XGB={opt_res["best_w_xgb"]:.4f}')

## 3. Held-Out Evaluation on 4 Approaches

In [ ]:
comparison = compare_four_approaches(rf, xgb, hybrid, X_test, y_test, metadata['class_names'], high_risk_idx=1)
comp_df = pd.DataFrame([{
    'Model': c['model_name'],
    'Accuracy (%)': c['accuracy'],
    'Macro F1 (%)': c['macro_f1'],
    'Anemia Sensitivity (%)': c['per_class']['Anemic']['recall'],
    'ROC-AUC': c['roc_auc']
} for c in comparison])
display(comp_df)